# Praktikum 1: Klasifikasi Gambar (Day vs Night) dengan Neural Network & HOG
**Nama:** M Nawfal M A  
**NIM:** 244107020174  
**Kelas:** Pembelajaran Mesin - Minggu 15

## Langkah 1 - Import Library

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pickle

from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models

## Langkah 2 - Load Dataset
*(Catatan: Letakkan folder `images/training` dan `images/test` di dalam folder `Dataset/`)*

In [4]:
def load_dataset(img_dir):
    p = Path(img_dir)
    img_list = []
    for folder in p.glob('*'):
        label = folder.name
        for file in folder.glob('*.jpg'):
            img = cv2.imread(str(file))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_list.append((img, label))
    return img_list

train_dir = "Dataset/images/training/"
test_dir  = "Dataset/images/test/"

try:
    train_img = load_dataset(train_dir)
    test_img  = load_dataset(test_dir)
    print(f"Jumlah data training: {len(train_img)}")
    print(f"Jumlah data testing: {len(test_img)}")
except Exception as e:
    print("Dataset belum ada di folder Dataset/images/")

Jumlah data training: 240
Jumlah data testing: 160


## Langkah 3 & 4 - Preprocessing & Ekstraksi Fitur HOG

In [6]:
def resize_image(img, size=(256,256)):
    return cv2.resize(img, size)

def label_encoder(label):
    return 1 if label == 'day' else 0

def preprocess(img_list):
    X = []
    y = []
    for img, label in img_list:
        img_std = resize_image(img)
        X.append(img_std)
        y.append(label_encoder(label))
    return X, y

def extract_hog(X_imgs):
    feats = []
    for img in X_imgs:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        hog_feat = hog(gray,
                       orientations=9,
                       pixels_per_cell=(8,8),
                       cells_per_block=(2,2),
                       block_norm='L2-Hys',
                       visualize=False,
                       feature_vector=True)
        feats.append(hog_feat)
    return np.array(feats)

## Langkah 5 - 7: Normalisasi & Split Data

In [7]:
# Jalankan jika train_img dan test_img sudah berhasil di-load
if 'train_img' in locals() and len(train_img) > 0:
    X_train_img, y_train = preprocess(train_img)
    X_test_img,  y_test  = preprocess(test_img)

    X_train_feat = extract_hog(X_train_img)
    X_test_feat  = extract_hog(X_test_img)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_feat)
    X_test_scaled  = scaler.transform(X_test_feat)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=42
    )

    y_train = np.array(y_train)
    y_val = np.array(y_val)
    y_test = np.array(y_test)

## Langkah 8 - 10: Membangun, Training & Evaluasi Neural Network

In [8]:
if 'X_train' in locals():
    input_dim = X_train.shape[1]

    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()

    history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_val, y_val))

    test_loss, test_acc = model.evaluate(X_test_scaled, y_test)
    print("\nAkurasi Test:", test_acc)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │     4,428,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,436,737 (16.92 MB)

 Trainable params: 4,436,737 (16.92 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 149ms/step - accuracy: 0.7656 - loss: 1.1518 - val_accuracy: 1.0000 - val_loss: 0.0040
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.9896 - loss: 0.0264 - val_accuracy: 1.0000 - val_loss: 6.6180e-04
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 1.0000 - loss: 5.7111e-06 - val_accuracy: 0.9792 - val_loss: 0.0372
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 1.0000 - loss: 3.6534e-05 - val_accuracy: 0.9792 - val_loss: 0.0884
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 1.0000 - loss: 5.8667e-05 - val_accuracy: 0.9792 - val_loss: 0.1167
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 1.0000 - loss: 3.3181e-05 - val_accuracy: 0.9792 - val_loss: 0.1317
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 1.0000 - loss: 1.5412e-05 - val_accuracy: 0.9792 - val_loss: 0.1386
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 1.0000 - loss: 8.7862e-06 - val_accurac

## Langkah 12: Menyimpan Model dan Scaler (Export Artifacts)

In [9]:
if 'model' in locals():
    # Simpan Model dan Scaler ke folder lokal
    model.save('day_night_model.h5')
    print("✅ Model berhasil disimpan sebagai: day_night_model.h5")

    with open('scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    print("✅ Scaler berhasil disimpan sebagai: scaler.pkl")

✅ Model berhasil disimpan sebagai: day_night_model.h5
✅ Scaler berhasil disimpan sebagai: scaler.pkl
